# Giai đoạn 3 — Tối ưu DSP nhúng & Kiểm chứng STM32

- Xuất model .tflite & chuẩn bị input mẫu

In [11]:
# notebooks/giai_doan_3/01_export_model.ipynb
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from common.models import build_mlp, compile_classifier
from common.quantization import quantize_model_int8
from common.training import iterate_lolo_splits
from common.mcu_export import export_tflite_for_mcu, save_signal_sample_csv, print_manual_workflow_notes

# Load
feature_df = pd.read_csv("outputs/tables/feature_table_full.csv")
feature_cols = [c for c in feature_df.columns 
                if c.startswith(("time_", "order_", "envelope_"))]
for c in feature_cols:
    feature_df[c] = pd.to_numeric(feature_df[c], errors='coerce')

print(f"Features: {len(feature_cols)}, Files: {len(feature_df)}")
    

# Fold test_load=0
for fold_info, train_df, val_df, test_df in iterate_lolo_splits(feature_df):
    if fold_info["test_load"] == 0:
        break

fit_df = pd.concat([train_df, val_df]) # type: ignore
X_train = fit_df[feature_cols].values.astype(np.float64)
y_train = np.asarray(fit_df["label"])

# Scale + encode
scaler = StandardScaler()
label_enc = LabelEncoder()
X_train_s = scaler.fit_transform(X_train)
y_train_enc = np.asarray(label_enc.fit_transform(y_train), dtype=np.int32)

# Train MLP (32 features)
model = build_mlp(input_dim=32)
compile_classifier(model)
model.fit(X_train_s, y_train_enc, epochs=200, batch_size=8, verbose=1)

# Quantize INT8
tflite_bytes = quantize_model_int8(model, X_train_s, n_representative_samples=20)

# Export
export_tflite_for_mcu(tflite_bytes, "outputs/models/mlp_32_int8.tflite",
                       model_tag="MLP_32_INT8")
save_signal_sample_csv(X_train_s, "outputs/models/test_input_samples.csv", n_samples=20)
print_manual_workflow_notes()

Features: 32, Files: 40
Epoch 1/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 0.2000 - loss: 1.7773
Epoch 2/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - accuracy: 0.2333 - loss: 1.6172 
Epoch 3/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.2333 - loss: 1.4870 
Epoch 4/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.3000 - loss: 1.3795 
Epoch 5/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.4667 - loss: 1.2643 
Epoch 6/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6000 - loss: 1.1711 
Epoch 7/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.6667 - loss: 1.0902 
Epoch 8/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.7333 - loss: 1.0198 
Epoch 9/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - accuracy: 0.8000 - loss: 0.9581 
Epoch 10/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.8000 - loss: 0.8957 
Epoch 11/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.8000 - loss: 0.8466
Epoch 12/200
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step 

INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpexwj8tqc\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpexwj8tqc'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2328371278480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328557469328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558716496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558716880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558716304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558715344: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Đã lưu [MLP_32_INT8]: F:\CODE\NCKH_TinyML\TinyML_CWRU\notebooks\giai_doan_0\outputs\models\mlp_32_int8.tflite (5672 bytes)
Đã lưu 20 mẫu vào: F:\CODE\NCKH_TinyML\TinyML_CWRU\notebooks\giai_doan_0\outputs\models\test_input_samples.csv

Có 3 cách lấy số đo Flash/RAM/MACC thật cho Bảng 3a/3b (mục 4.1), xếp theo
độ dễ dùng (không phải độ chính xác — cả 3 cho cùng 1 nguồn số liệu):

1) WEB UI (dễ nhất, không cài gì) — https://stedgeai-dc.st.com/
   - Đăng nhập bằng tài khoản myST (bạn đã có quyền truy cập).
   - Kéo-thả file .tflite xuất từ export_tflite_for_mcu().
   - Chọn STM32 target (F4/G4/H7...), chạy "Analyze" (static analysis —
     bắt buộc theo mục 3.2) để lấy Flash/RAM/MACC.
   - Tùy chọn: chạy "Benchmark" (board farm) để lấy latency thật trên
     board vật lý — thực nghiệm MỞ RỘNG, không bắt buộc (mục 0.4).
   - Xuất báo cáo (nút export trên UI) — dán dòng tóm tắt vào
     parse_stedgeai_cli_summary() ở trên, hoặc đọc trực tiếp trên UI.

2) CLI `stedgeai` (nếu cài STM32Cube.AI 

In [12]:
tflite_bytes = quantize_model_int8(model, X_train_s, n_representative_samples=20)
export_tflite_for_mcu(tflite_bytes, "outputs/models/mlp_32_int8.tflite",
                       model_tag="MLP_32_INT8")
save_signal_sample_csv(X_train_s, "outputs/models/test_input_samples.csv", n_samples=20)
print_manual_workflow_notes()

INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpoy7g1xg2\assets


INFO:tensorflow:Assets written to: C:\Users\Admin\AppData\Local\Temp\tmpoy7g1xg2\assets


Saved artifact at 'C:\Users\Admin\AppData\Local\Temp\tmpoy7g1xg2'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32), dtype=tf.float32, name='features')
Output Type:
  TensorSpec(shape=(None, 4), dtype=tf.float32, name=None)
Captures:
  2328371278480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328557469328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558716496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558716880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558716304: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2328558715344: TensorSpec(shape=(), dtype=tf.resource, name=None)


f:\CODE\NCKH_TinyML\TinyML_CWRU\.venv\Lib\site-packages\tensorflow\lite\python\convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(


Đã lưu [MLP_32_INT8]: F:\CODE\NCKH_TinyML\TinyML_CWRU\notebooks\giai_doan_0\outputs\models\mlp_32_int8.tflite (5672 bytes)
Đã lưu 20 mẫu vào: F:\CODE\NCKH_TinyML\TinyML_CWRU\notebooks\giai_doan_0\outputs\models\test_input_samples.csv

Có 3 cách lấy số đo Flash/RAM/MACC thật cho Bảng 3a/3b (mục 4.1), xếp theo
độ dễ dùng (không phải độ chính xác — cả 3 cho cùng 1 nguồn số liệu):

1) WEB UI (dễ nhất, không cài gì) — https://stedgeai-dc.st.com/
   - Đăng nhập bằng tài khoản myST (bạn đã có quyền truy cập).
   - Kéo-thả file .tflite xuất từ export_tflite_for_mcu().
   - Chọn STM32 target (F4/G4/H7...), chạy "Analyze" (static analysis —
     bắt buộc theo mục 3.2) để lấy Flash/RAM/MACC.
   - Tùy chọn: chạy "Benchmark" (board farm) để lấy latency thật trên
     board vật lý — thực nghiệm MỞ RỘNG, không bắt buộc (mục 0.4).
   - Xuất báo cáo (nút export trên UI) — dán dòng tóm tắt vào
     parse_stedgeai_cli_summary() ở trên, hoặc đọc trực tiếp trên UI.

2) CLI `stedgeai` (nếu cài STM32Cube.AI 

In [13]:
import timeit
import numpy as np
from common import dsp

fs = 12000
x = np.random.randn(2048)  # một đoạn tín hiệu

# Square-Law
def square_law():
    return dsp.square_law_envelope(x, fs, band=(2500,3500), lp_cutoff=500)

# Hilbert
def hilbert_method():
    return dsp.hilbert_envelope(x, fs, band=(2500,3500))

t_sq = timeit.timeit(square_law, number=1000) / 1000 * 1000  # ms
t_hil = timeit.timeit(hilbert_method, number=1000) / 1000 * 1000
print(f"Square-Law: {t_sq:.3f} ms")
print(f"Hilbert:    {t_hil:.3f} ms")

Square-Law: 0.894 ms
Hilbert:    0.653 ms
